<a href="https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My lane as an ML task (type)

I chose **ranking** as my ML task.

The goal is to rank web pages by their priority for content-refresh review. Instead of making only a yes/no prediction about whether a page needs attention, the output should produce an ordered list so an SEO or content team can focus on the highest-priority pages first.

Ranking fits this problem because the practical decision is about prioritization. A team has limited time and cannot review every page at once, so the useful output is a ranked list that supports deciding which pages to review first.

In [2]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "/content/flyrank-ml-internship-starter"

# Clone the starter repository if it is not already available
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

# Check that the starter dataset exists
data_path = "data/raw/content_refresh_anonymized.csv"
print("Dataset exists:", os.path.exists(data_path))

assert os.path.exists(data_path), "Starter dataset was not found."

# Load the dataset
import pandas as pd

df = pd.read_csv(data_path)

print("\nDataset shape:", df.shape)
print("\nColumns available for the ranking task:")

relevant_columns = [
    "trend_direction",
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days"
]

for column in relevant_columns:
    if column in df.columns:
        print("✓", column)

Working directory: /content/flyrank-ml-internship-starter
Dataset exists: True

Dataset shape: (30000, 44)

Columns available for the ranking task:
✓ trend_direction
✓ search_volume
✓ impressions_90d
✓ ctr
✓ avg_position
✓ word_count
✓ content_age_days


## Target or proxy

I will use **trend_direction** as the target proxy for the ranking task.

The target represents the observed direction of a page's performance in the starter dataset, with categories such as down, stable, up, new, and flat. I will use this observed outcome to learn patterns that can help prioritize pages for content-refresh review.

This is a proxy rather than a direct measure of whether a page should be refreshed. The actual business decision is which pages deserve attention first, while the observed trend provides a measurable signal that can support that decision.

In [3]:
print("Target column:", "trend_direction")
print("\nObserved target distribution:")
print(df["trend_direction"].value_counts())

print("\nNumber of unique target categories:", df["trend_direction"].nunique())

Target column: trend_direction

Observed target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Number of unique target categories: 5


## Success metric

I will use **Precision@50** as the main success metric.

Precision@50 measures how many of the top 50 pages selected by the ranking are relevant according to the evaluation target. This fits the decision because a content or SEO team has limited time and is likely to act on a small number of high-priority recommendations first.

A higher Precision@50 means that more of the pages near the top of the ranking are relevant to the target. In the starter experiment, the Random Forest achieved **0.740 Precision@50**, compared with **0.240** for the hand-written baseline. This suggests that the learned ranking was more useful than the fixed baseline on this dataset.

In [6]:
import os
import json

results_path = "outputs/model_results.json"

if not os.path.exists(results_path):
    print("model_results.json not found.")
    print("Run the starter pipeline first to generate the model results.")
else:
    results = json.load(open(results_path))

    baseline = results["baseline"]["baseline_precision_at_50"]
    random_forest = results["models"]["random_forest"]["precision_at_50"]

    print(f"Hand-written baseline Precision@50: {baseline:.3f}")
    print(f"Random Forest Precision@50: {random_forest:.3f}")
    print(f"Difference: {random_forest - baseline:.3f}")

Hand-written baseline Precision@50: 0.240
Random Forest Precision@50: 0.740
Difference: 0.500


## The unit of analysis, as a real dataframe

The unit of analysis is **one web page**.

Each row represents one page and contains observed information about that page, including search volume, impressions, CTR, average position, word count, content age, and trend direction.

The ranking model will use page-level observations to estimate which pages should receive higher priority for content-refresh review.

In [7]:
print("Dataset shape:", df.shape)
print("\nOne row represents one web page.")
print("\nExample rows:")

df.head(5)

Dataset shape: (30000, 44)

One row represents one web page.

Example rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## Why ML beats a fixed rule here

A fixed rule could prioritize pages using one simple condition, such as reviewing every page with low CTR or low impressions. However, page performance is influenced by several signals at the same time, including search volume, impressions, CTR, average position, word count, and content age.

These signals can interact in ways that are difficult to capture with one or two if-statements. A machine learning model can learn patterns across multiple features and produce a ranking based on those combined signals.

The starter experiment provides directional evidence for this approach: the Random Forest achieved **0.740 Precision@50**, compared with **0.240** for the hand-written baseline. This suggests that, on this dataset and evaluation setup, combining multiple signals can produce a more useful prioritization than the fixed baseline.

In [8]:
important_features = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days"
]

print("Available features that could support the ranking:")

for feature in important_features:
    if feature in df.columns:
        print("✓", feature)

print("\nNumber of candidate features available:",
      sum(feature in df.columns for feature in important_features))

Available features that could support the ranking:
✓ search_volume
✓ impressions_90d
✓ ctr
✓ avg_position
✓ word_count
✓ content_age_days

Number of candidate features available: 6


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.